In [29]:
from mpqp.core.circuit import CircuitBinding
from mpqp.core.circuit import BindingMode, QCircuit
from mpqp.core.instruction.measurement.basis_measure import BasisMeasure
from mpqp.core.instruction.measurement.expectation_value import (
    ExpectationMeasure,
    Observable,
)
from mpqp.core.instruction.measurement import pX, pY, pZ, pI
from mpqp.execution.result import BatchResult, StateVector
from sympy import Symbol
from mpqp.gates import *

theta, phi, psi = Symbol('θ'), Symbol('phi'), Symbol('psi')
a = Symbol("a")
c1 = QCircuit([U(theta, phi, psi, 0)], label="c1")
c2 = QCircuit([H(0)], label="c2")
c3 = QCircuit([H(0), CNOT(0, 1), Ry(theta, 0)], label="c3")
c4 = QCircuit([H(0), H(1), CNOT(0, 1)], label="c4")

v1 = {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
v2 = {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
v3 = {'θ': 3.0, 'phi': 3.0, 'psi': 3.0}
v4 = {'θ': 4.0, 'phi': 4.0, 'psi': 4.0}

m1 = ExpectationMeasure(
    Observable(pI), label="Exp1", shots=2024, optimize_measurement=False
)
m2 = ExpectationMeasure(
    Observable(pX @ pZ), label="Exp2", shots=2024, optimize_measurement=False
)
m3 = BasisMeasure(label="b3", shots=2024)
m4 = None

m_I = ExpectationMeasure(Observable(pI), label="Exp_I", shots=2024)
m_Z = ExpectationMeasure(
    Observable(pZ), label="Exp_Z", shots=2024, optimize_measurement=False
)


def zip_tests_observable():
    cb_zip = CircuitBinding(
        circuits=c3,
        values=[v1, v2, v3],
        measurements=[m1, m2, m_Z],
        mode=BindingMode.ZIP,
    )

    def val_zip(res: BatchResult):
        assert isinstance(res, BatchResult)
        print(len(res))
        for r in res.results:
            val = (
                list(r.expectation_values.values())[0]
                if isinstance(r.expectation_values, dict)
                else r.expectation_values
            )
            print(round(val, 5) == 1.0)

    return [(cb_zip, val_zip)]

In [30]:
from mpqp.execution.devices import AWSDevice, IBMDevice
from mpqp.execution.runner import run

for cb, val in zip_tests_observable():
    res = run(cb, AWSDevice.BRAKET_LOCAL_SIMULATOR)
    r2 = run(cb, IBMDevice.AER_SIMULATOR)
    print(res)
    print()
    print(r2)

BatchResult: 3 results
    Result: c3, AWSDevice, BRAKET_LOCAL_SIMULATOR
    Variables' values: {'θ': [1.0], 'phi': [1.0], 'psi': [1.0]}
                    Expectation value: 1.0
                    Error/Variance: None
    With observables: [pI@pI]
    

    Result: c3, AWSDevice, BRAKET_LOCAL_SIMULATOR
    Variables' values: {'θ': [2.0], 'phi': [2.0], 'psi': [2.0]}
                    Expectation value: 0.9179841897233202
                    Error/Variance: None
    With observables: [pX@pZ]
    

    Result: c3, AWSDevice, BRAKET_LOCAL_SIMULATOR
    Variables' values: {'θ': [3.0], 'phi': [3.0], 'psi': [3.0]}
                    Expectation value: -0.010869565217391304
                    Error/Variance: None
    With observables: [pZ@pI]
    


BatchResult: 3 results
    Result: c3, IBMDevice, AER_SIMULATOR
      Exp1_0:
        Expectation value: 1.0
        Error/Variance: 0.0
      Exp2_0:
        Expectation value: -0.8528395061728395
        Error/Variance: 0.01160384731241371

In [26]:
print(cb.to_other_device(IBMDevice.AER_SIMULATOR))

[((<qiskit.circuit.quantumcircuit.QuantumCircuit object at 0x0000020037618D10>, [[SparsePauliOp(['II'],
              coeffs=[1.+0.j])], [SparsePauliOp(['XZ'],
              coeffs=[1.+0.j])], [SparsePauliOp(['ZI'],
              coeffs=[1.+0.j])]], [[1.0], [2.0], [3.0]]), Job(JobType.OBSERVABLE, QCircuit([H(0), CNOT(0, 1), Ry(θ, 0), ExpectationMeasure(Observable(pI@pI, 'Exp1_0'), [0, 1], shots=2024, label='Exp1'), ExpectationMeasure(Observable(pX@pZ, 'Exp2_0'), shots=2024, label='Exp2'), ExpectationMeasure(Observable(pZ@pI, 'Exp_Z_0'), [0, 1], shots=2024, label='Exp_Z')], label="c3"), IBMDevice.AER_SIMULATOR))]


In [17]:
cb = c3 + QCircuit([m2])
print(run(cb, IBMDevice.AER_SIMULATOR))
print(run(cb, AWSDevice.BRAKET_LOCAL_SIMULATOR))

Result: c3, IBMDevice, AER_SIMULATOR
                Expectation value: 0.02419753086419753
                Error/Variance: 0.022215715486293457
Result: c3, AWSDevice, BRAKET_LOCAL_SIMULATOR
                Expectation value: -0.018774703557312252
                Error/Variance: None


In [ ]:
from mpqp.core.languages import Language

c3 = QCircuit([H(0), CNOT(0, 1), Rx(theta, 0), Ry(phi, 0)], label="c3")

cc = c3.to_other_language(Language.QISKIT)
data = [str(p) for p in cc.parameters.data]
print(data)

['2*phi', 'θ']
